# 02 · Bronze: land raw files as Delta
* Reads every file in the landing folders (`firms/*.csv`, `epa/*.json`, `noaa/*.json`).
* Keeps every source value **as a string** (no cleaning here), and adds metadata:
  `_source`, `_source_file`, `_data_date` (the day the data is for), `_ingested_at_utc`, `_batch_id`, `_content_hash`.
* **Incremental:** a file whose name and content hash are already in `ops_ingest_manifest` is skipped.
  A file with the same name but a new hash (the source revised that day) is loaded again as a new version;
  Silver then keeps the latest version.
* Bronze is append-only, so the full history of what was received stays auditable.

In [0]:
%run ./00_config

# 00 · Config
Shared settings for every notebook. Every other notebook loads this one with `%run ./00_config`.

**Before the first run:** change `CATALOG` / `SCHEMA` if needed, then run `01_setup` once to create the schema and volume.

Config loaded: workspace.wildfire, raw files in /Volumes/workspace/wildfire/raw


In [0]:
import hashlib, os, re, uuid

dbutils.widgets.text("only_date", "", "Only ingest files for this date (YYYY-MM-DD, blank = all)")
ONLY_DATE = dbutils.widgets.get("only_date").strip()
BATCH_ID = dt.datetime.utcnow().strftime("%Y%m%dT%H%M%S") + "-" + uuid.uuid4().hex[:6]
DATE_RE = re.compile(r"(\d{4}-\d{2}-\d{2})")

def file_hash(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def already_loaded():
    if not spark.catalog.tableExists(t("ops_ingest_manifest")):
        return set()
    return {(r.file_name, r.content_hash) for r in spark.table(t("ops_ingest_manifest")).select("file_name", "content_hash").collect()}

def new_files(source, ext):
    folder = LANDING[source]
    seen = already_loaded()
    out = []
    for name in sorted(os.listdir(folder)):
        if not name.endswith(ext):
            continue
        m = DATE_RE.search(name)
        if not m or (ONLY_DATE and m.group(1) != ONLY_DATE):
            continue
        path = f"{folder}/{name}"
        h = file_hash(path)
        if (name, h) in seen:
            print(f"skip (already loaded): {name}")
            continue
        out.append((path, name, m.group(1), h))
    return out

def add_metadata(df, source, path, data_date, h):
    return (df.select([F.col(c).cast("string").alias(c) for c in df.columns])
              .withColumn("_source", F.lit(source))
              .withColumn("_source_file", F.lit(path))
              .withColumn("_data_date", F.to_date(F.lit(data_date)))
              .withColumn("_ingested_at_utc", F.current_timestamp())
              .withColumn("_batch_id", F.lit(BATCH_ID))
              .withColumn("_content_hash", F.lit(h)))

def record_manifest(source, path, name, data_date, h, rows):
    spark.createDataFrame(
        [(source, path, name, dt.date.fromisoformat(data_date), h, int(rows), dt.datetime.utcnow(), BATCH_ID)],
        "source string, file_path string, file_name string, data_date date, content_hash string, rows_loaded long, ingested_at_utc timestamp, batch_id string",
    ).write.mode("append").saveAsTable(t("ops_ingest_manifest"))

def append_bronze(df, table):
    df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(t(table))

/home/spark-1e3dde31-43c9-4c36-888f-0f/.ipykernel/63/command-8793491404667951-2870540328:5: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  BATCH_ID = dt.datetime.utcnow().strftime("%Y%m%dT%H%M%S") + "-" + uuid.uuid4().hex[:6]


## NASA FIRMS (CSV)

In [0]:
started = time.time(); total = 0
for path, name, d, h in new_files("firms", ".csv"):
    raw = spark.read.option("header", True).option("inferSchema", False).csv(path)
    df = add_metadata(raw, "nasa_firms", path, d, h)
    n = df.count()
    append_bronze(df, "bronze_firms")
    record_manifest("nasa_firms", path, name, d, h, n)
    total += n
log_step("bronze_firms", "bronze", total, started, {"batch_id": BATCH_ID})

/home/spark-1e3dde31-43c9-4c36-888f-0f/.ipykernel/63/command-8793491404667951-2870540328:49: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  [(source, path, name, dt.date.fromisoformat(data_date), h, int(rows), dt.datetime.utcnow(), BATCH_ID)],
/home/spark-1e3dde31-43c9-4c36-888f-0f/.ipykernel/63/command-6312931087636986-1423734058:45: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  row = [(step, layer, int(rows), float(elapsed), dt.datetime.utcnow(), json.dumps(extra or {}))]


[bronze] bronze_firms: 33,714 rows in 83.9s


## EPA AQS daily PM2.5 (JSON API response: `{"Header": [...], "Data": [...]}`)

In [0]:
started = time.time(); total = 0
for path, name, d, h in new_files("epa", ".json"):
    raw = spark.read.option("multiLine", True).json(path)
    from pyspark.sql.types import ArrayType, StructType
    dtype = raw.schema["Data"].dataType if "Data" in raw.columns else None
    if not (isinstance(dtype, ArrayType) and isinstance(dtype.elementType, StructType)):
        # AQS answers "No data matched your selection" with an empty Data array
        print(f"no records in {name}; recorded with 0 rows"); record_manifest("epa_aqs", path, name, d, h, 0); continue
    rows = raw.select(F.explode("Data").alias("r")).select("r.*")
    df = add_metadata(rows, "epa_aqs", path, d, h)
    n = df.count()
    append_bronze(df, "bronze_epa_pm25")
    record_manifest("epa_aqs", path, name, d, h, n)
    total += n
log_step("bronze_epa", "bronze", total, started, {"batch_id": BATCH_ID})

/home/spark-1e3dde31-43c9-4c36-888f-0f/.ipykernel/63/command-8793491404667951-2870540328:49: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  [(source, path, name, dt.date.fromisoformat(data_date), h, int(rows), dt.datetime.utcnow(), BATCH_ID)],
/home/spark-1e3dde31-43c9-4c36-888f-0f/.ipykernel/63/command-6312931087636986-1423734058:45: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  row = [(step, layer, int(rows), float(elapsed), dt.datetime.utcnow(), json.dumps(extra or {}))]


[bronze] bronze_epa: 7,457 rows in 38.3s


## NOAA GHCN-Daily (JSON: `{"results": [...]}`, pages already combined by the downloader)

In [0]:
started = time.time(); total = 0
for path, name, d, h in new_files("noaa", ".json"):
    raw = spark.read.option("multiLine", True).json(path)
    from pyspark.sql.types import ArrayType, StructType
    dtype = raw.schema["results"].dataType if "results" in raw.columns else None
    if not (isinstance(dtype, ArrayType) and isinstance(dtype.elementType, StructType)):
        record_manifest("noaa_cdo", path, name, d, h, 0); continue
    rows = raw.select(F.explode("results").alias("r")).select("r.*")
    df = add_metadata(rows, "noaa_cdo", path, d, h)
    n = df.count()
    append_bronze(df, "bronze_noaa_daily")
    record_manifest("noaa_cdo", path, name, d, h, n)
    total += n
log_step("bronze_noaa", "bronze", total, started, {"batch_id": BATCH_ID})

/home/spark-1e3dde31-43c9-4c36-888f-0f/.ipykernel/63/command-8793491404667951-2870540328:49: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  [(source, path, name, dt.date.fromisoformat(data_date), h, int(rows), dt.datetime.utcnow(), BATCH_ID)],
/home/spark-1e3dde31-43c9-4c36-888f-0f/.ipykernel/63/command-6312931087636986-1423734058:45: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  row = [(step, layer, int(rows), float(elapsed), dt.datetime.utcnow(), json.dumps(extra or {}))]


[bronze] bronze_noaa: 15,184 rows in 31.8s


## Reference data (overwritten each run: small, slowly changing)

In [0]:
counties = spark.read.option("header", True).csv(LANDING["counties"])
counties.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(t("bronze_ref_counties"))

stations = (spark.read.option("multiLine", True).json(LANDING["noaa_stations"])
                 .select(F.explode("results").alias("r")).select("r.*"))
stations.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(t("bronze_ref_noaa_stations"))

display(spark.table(t("ops_ingest_manifest")).orderBy(F.desc("ingested_at_utc")).limit(20))

source,file_path,file_name,data_date,content_hash,rows_loaded,ingested_at_utc,batch_id
noaa_cdo,/Volumes/workspace/wildfire/raw/noaa/noaa_2020-08-21.json,noaa_2020-08-21.json,2020-08-21,057e6d86e62bc623bc8e09fa1141ca0e6cd8862d977042c5b50b2c45961b7c3c,2160,2026-10-29T20:44:30.631Z,20261029T204153-f87fd9
noaa_cdo,/Volumes/workspace/wildfire/raw/noaa/noaa_2020-08-20.json,noaa_2020-08-20.json,2020-08-20,695ab3600cf7d9e391dab5294082c381f7ddabe5f8a9271693601a09712d389a,2178,2026-10-29T20:44:26.906Z,20261029T204153-f87fd9
noaa_cdo,/Volumes/workspace/wildfire/raw/noaa/noaa_2020-08-19.json,noaa_2020-08-19.json,2020-08-19,9e26ea45d34ab12d97868e369d627c67b21041eb0d3d58b686aaa8f8d272cc7b,2168,2026-10-29T20:44:22.706Z,20261029T204153-f87fd9
noaa_cdo,/Volumes/workspace/wildfire/raw/noaa/noaa_2020-08-18.json,noaa_2020-08-18.json,2020-08-18,fbe2c1b8217a9a261df9e35dd88d00ca8374b3ae35f5b50c94357580d7484d9a,2195,2026-10-29T20:44:18.848Z,20261029T204153-f87fd9
noaa_cdo,/Volumes/workspace/wildfire/raw/noaa/noaa_2020-08-17.json,noaa_2020-08-17.json,2020-08-17,361a300c5535746e7d34fc6e80bf6118d0bc1a68ff4615ba1c0f6ff71586ce69,2171,2026-10-29T20:44:14.933Z,20261029T204153-f87fd9
noaa_cdo,/Volumes/workspace/wildfire/raw/noaa/noaa_2020-08-16.json,noaa_2020-08-16.json,2020-08-16,9f0083f16b27f14629c8bd63957b415df4b4cd8c88b886e7fbaa300e7d181f84,2177,2026-10-29T20:44:10.631Z,20261029T204153-f87fd9
noaa_cdo,/Volumes/workspace/wildfire/raw/noaa/noaa_2020-08-15.json,noaa_2020-08-15.json,2020-08-15,41df8f5eea4ebba1836ff612fc7686c3213091af358cab60118262f329c8bcc0,2135,2026-10-29T20:44:06.486Z,20261029T204153-f87fd9
epa_aqs,/Volumes/workspace/wildfire/raw/epa/epa_2020-08-21.json,epa_2020-08-21.json,2020-08-21,7932ca1941bc48b63c1e3abf3f17ea543749f229e4f1cb90d5332203c0c64d70,1047,2026-10-29T20:43:57.143Z,20261029T204153-f87fd9
epa_aqs,/Volumes/workspace/wildfire/raw/epa/epa_2020-08-20.json,epa_2020-08-20.json,2020-08-20,81c19fabc4262a41a6d922f177db8db1ea337bf48cc6e332cd75bf81c285accd,1025,2026-10-29T20:43:52.774Z,20261029T204153-f87fd9
epa_aqs,/Volumes/workspace/wildfire/raw/epa/epa_2020-08-19.json,epa_2020-08-19.json,2020-08-19,e270b205c6538fc0175e0b5344c69678c8c7d093193a207663d369ba0f08bdc8,1383,2026-10-29T20:43:48.368Z,20261029T204153-f87fd9


In [0]:
for tb in ["bronze_firms", "bronze_epa_pm25", "bronze_noaa_daily"]:
    print(tb, spark.table(f"workspace.wildfire.{tb}").count())
spark.table("workspace.wildfire.ops_ingest_manifest").groupBy("source").count().show()

bronze_firms 33714
bronze_epa_pm25 7457
bronze_noaa_daily 15184
+----------+-----+
|    source|count|
+----------+-----+
|nasa_firms|    8|
|  noaa_cdo|    7|
|   epa_aqs|    7|
+----------+-----+

